# Circuit Quantization Using Method of Nodes

In [1]:
using ProjectRoot
using Pkg
const ROOT = @projectroot 
Pkg.activate(ROOT)


using Revise
using XoniumUnitCell
using LinearAlgebra
using Symbolics

  Activating project at `~/Projects/xonium/XoniumUnitCell.jl`


## FF Direct Multipath Coupling

In this notebook, we will utilize `Symbolics.jl` along with a custom-built graph -> LinearAlgebra library to derive symbolic expressions for Hamiltonian parameters in terms of circuit parameters. We can then use the derived symbolic expressions to do sweeps over circuit parameters or even derive the response of the Hamiltonian properties (such as ZZ) as functions of the circuit parameters.

The Xonium are composed of outer rings and inner islands, conected by a JJ. There is an active node on each ring and island. The rings are intentionally capacitively coupled to each other, but due to parasitic capacitances the true graph is generally all-to-all connected.


 The Xonia will be labeled `A` and `B`. The rings will correspond to labels `r` and islands to labels `i`. 

The correspondance to the diagram in Abhishek's marimo notebooks is: the rings are the even nodes, the islands are the odd nodes.

Below we define the symbols for the various circuit parameters.


In [2]:
### capacitances and inductances
# Fluxonium A, CrA is Capacitance, Ring A. Similar wiht "i" for "Island"
@variables CrA CiA CriA LriA 
# Fluxonium B
@variables CrB CiB CriB LriB

# capacitive couplings, left index is A, right is B
# Crr is Capacitance Ring A -Ring B, Cri is Capacitance Ring A-Island B
@variables Crr Cii Cir Cri
### inductive couplings, same labeling rules
@variables Lrr;



Next we define the nodes that make up the capacitive graph. The node constructors take a symbolic label and a ground capacitance value, which is allowed to be a variable `Num`.

In [3]:
# Fluxonium A modes
rA, iA = CNode(:rA, CrA), CNode(:iA, CiA)
# Fluxonium B modes
rB, iB = CNode(:rB, CrB), CNode(:iB, CiB);

Next we define the edges of the graph. Each graph edge takes a tuple of symbols corresponding to the symbolic label of each node (`:rA`, for example), and two optional keyword arguments `C, L`, representing the capacitance and inductance across the edge connecting the two nodes.

In [4]:
# Inter-qubit edges. riA is Ring-Island edge Fluxonium A
riA, riB = (
    CEdge(:rA, :iA; C=CriA, L=LriA),
    CEdge(:rB, :iB; C=CriB, L=LriB)
)
# Ring A -Ring B, Island A -Island B
rr, ii = (
    CEdge(:rA, :rB; C=Crr, L=Lrr),
    CEdge(:iA, :iB; C=Cii) # no island-island inductance
)

# Ring A -Island B, Island A - Ring B 
ri, ir = (
    CEdge(:rA, :iB; C=Cri),
    CEdge(:iA, :rB; C=Cir),
);

Finally we construct the nodal graph of the Capacitive spanning tree.

In [5]:
nodes = [iA, rA, iB, rB]
edges = [
    riA, riB,
    rr, ii,
    ri, ir
]

# `simplify` cleans up the symbolic expressions of the matrices, nodes and edges,
# including those produced by basis transformations below
circuit = CGraph(nodes, edges; simplify = x -> Symbolics.simplify(x; expand=true));

The circuit-graph will compute the Maxwell capacitive and inverse inductive matrices from the graph. Each node is mapped to a basis vector in the basis of the capacitive and inductive matrices. The correspondence of node-to-vector is done by the ordering of the nodes in the vector given to the `CGraph` constructor.

In [6]:
display(basis_vectors(circuit))
display(circuit.C)
display(circuit.L_inv)

(iA = [1, 0, 0, 0], rA = [0, 1, 0, 0], iB = [0, 0, 1, 0], rB = [0, 0, 0, 1])

4×4 Matrix{Num}:
 CiA + Cii + Cir + CriA                   -CriA  …                    -Cir
                  -CriA  CrA + Cri + CriA + Crr                       -Crr
                   -Cii                    -Cri                      -CriB
                   -Cir                    -Crr     Cir + CrB + CriB + Crr

4×4 Matrix{Num}:
  1 / LriA                  -1 / LriA          0                          0
 -1 / LriA  (LriA + Lrr) / (LriA*Lrr)          0                   -1 / Lrr
         0                          0   1 / LriB                  -1 / LriB
         0                   -1 / Lrr  -1 / LriB  (LriB + Lrr) / (LriB*Lrr)

## Transformation to Difference and Sum Basis

The JJ's non-linear dynamics depend on the flux difference across the junction. Here, we transform into this basis. First we must construct the change-of-basis matrix into the unnormalized difference and sum modes. We write the new coordinates in terms of the old node fluxes,
$$
\vec{\psi} = T \vec{\phi}
$$

where $\vec{\phi}$ are the old node fluxes and $\vec{\psi}$ the new sum and difference coordinates, for example $\psi_{\Delta A} = \phi_{iA} - \phi_{rA}$, which is exactly the flux across the junction. Row $k$ of $T$ is the $k$-th new coordinate written in the old basis, so $T$ can be built from outer products of the new basis vectors with the old combinations.

The capacitive and inverse-inductive matrices are quadratic forms in the fluxes. Since $\vec{\phi} = T^{-1}\vec{\psi}$, they transform as $T^{-T} \, M \, T^{-1}$. This is the convention of the `xonium-taichi-couplers` marimo notebook, and it keeps the difference mode equal to the full junction phase. (Reading $T$ instead as $\vec{\phi} = T\vec{\psi}$ would give $T^{T} M T$, with a difference mode of half the junction phase, so $E_C$ would be four times too small.)

In [7]:
vecs = basis_vectors(circuit)
# new basis vector names
ΣA, ΔA, ΣB, ΔB = [vec for (_, vec) in pairs(vecs)]
new_vecs = (; ΣA, ΔA, ΣB, ΔB) # named tuple for later relabeling

# Row k of T writes the k-th new coordinate in terms of the old node fluxes, so ψ = T φ
T = (ΣA * (vecs.rA + vecs.iA)'
 + ΔA * (vecs.iA - vecs.rA)'
 + ΣB * (vecs.rB + vecs.iB)'
 + ΔB * (vecs.iB - vecs.rB)'
) .|> Rational{Int} # change eltype to Rational to make computations exact

4×4 Matrix{Rational{Int64}}:
 1   1  0   0
 1  -1  0   0
 0   0  1   1
 0   0  1  -1

The `CGraph` struct permits a basis transform as `M * graph * Q`, where M and Q are matrices. The return of this multiplication is a new graph, with the capacitivive and inductive maxwell-matrices transformed according to `M* C * Q`. The end result of the operation is checked for symmetry of both `C` and `L_inv`. If it is not symmetric up to a tolerance, it errors. If it is symmetric, the new graph's node and edge values are changed to match the capacitive and inductive matrices.

For the new coordinates $\vec{\psi} = T\vec{\phi}$ we therefore use $M = T^{-T}$ and $Q = T^{-1}$. The inverse is taken with exact rational entries, so no floating-point numbers appear in the symbolic expressions.

In [9]:
new_labels = (lbl for (lbl, _) in pairs(new_vecs)) |> collect
Tinv = inv(T) # exact inverse, so the entries stay rational
# φ = T⁻¹ψ, so the matrices transform as T⁻ᵀ M T⁻¹. The new graph is a separate variable so
# that re-running this cell does not transform the circuit twice.
circuit_ΣΔ = Tinv' * circuit * Tinv |> # do transformation
    circ -> relabel(circ, new_labels) # change node labels
display(circuit_ΣΔ.C)
display(circuit_ΣΔ.L_inv)
basis_vectors(circuit_ΣΔ)

4×4 Matrix{Num}:
          (1//4)*(CiA + Cii + Cir + CrA + Cri + Crr)  …             -(1//4)*(Cii + Cri) + (1//4)*(Cir + Crr)
 (1//4)*(CiA + Cii + Cir) - (1//4)*(CrA + Cri + Crr)                -(1//4)*(Cii + Crr) + (1//4)*(Cir + Cri)
                     (-1//4)*(Cii + Cir + Cri + Crr)     (1//4)*(CiB + Cii + Cri) - (1//4)*(Cir + CrB + Crr)
            -(1//4)*(Cii + Cri) + (1//4)*(Cir + Crr)       (1//4)*(CiB + Cii + Cir + CrB + Cri + Crr) + CriB

4×4 Matrix{Num}:
    1 / (4Lrr)                  -1 / (4Lrr)  …                 (1//4) / Lrr
   -1 / (4Lrr)  (LriA + 4Lrr) / (4LriA*Lrr)                   (-1//4) / Lrr
 (-1//4) / Lrr                 (1//4) / Lrr                     -1 / (4Lrr)
  (1//4) / Lrr                (-1//4) / Lrr     (LriB + 4Lrr) / (4LriB*Lrr)

(ΣA = [1, 0, 0, 0], ΔA = [0, 1, 0, 0], ΣB = [0, 0, 1, 0], ΔB = [0, 0, 0, 1])

The Hamiltonian is $H = \tfrac12\, \vec{q}^{\,T} C^{-1} \vec{q} + \tfrac12\, \vec{\psi}^{\,T} L^{-1} \vec{\psi} + \dots$, so the two kinds of parameters are read out differently:

- The inductive energy $E_L$ of a mode is the corresponding **diagonal element of the inverse-inductance matrix** `L_inv`, converted with `Linv2E_L`.
- The charging energy $E_C$ and the capacitive coupling $J$ need the **inverse of the capacitance matrix**, $C^{-1}$, which is what multiplies the charges. They are the diagonal and off-diagonal elements $(C^{-1})_{\Delta\Delta}$ and $(C^{-1})_{\Delta A,\Delta B}$, converted with `Cinv2E_C` and `Cinv2J`. A diagonal element of $C$ itself is not enough: the sum modes are capacitively coupled to the difference modes, so $(C^{-1})_{\Delta\Delta} \neq 1/C_{\Delta\Delta}$, and the sum modes cannot be discarded before inverting.

In [14]:
Cinv = inv(circuit_ΣΔ.C) # E_C and J come from the INVERSE capacitance matrix

ECA = ΔA' * Cinv * ΔA |> Cinv2E_C
ECB = ΔB' * Cinv * ΔB |> Cinv2E_C
ELA = ΔA' * circuit_ΣΔ.L_inv * ΔA |>  Linv2E_L
ELB = ΔB' * circuit_ΣΔ.L_inv * ΔB |>  Linv2E_L
JAB = ΔA' * Cinv * ΔB |> Cinv2J

(; ECA, ECB, ELA, ELB, JAB)

(ECA = (1.2834849832677848e-32((1//4)*(CiA + Cii + Cir + CrA + Cri + Crr)*(-(((1//4)*(CiB + Cii + Cri) - (1//4)*(Cir + CrB + Crr))^2) + (1//4)*((1//4)*(CiB + Cii + Cir + CrB + Cri + Crr) + CriB)*(CiB + Cii + Cir + CrB + Cri + Crr)) + (-(1//4)*((1//4)*(CiB + Cii + Cri) - (1//4)*(Cir + CrB + Crr))*(Cii + Cir + Cri + Crr) - (1//4)*(CiB + Cii + Cir + CrB + Cri + Crr)*(-(1//4)*(Cii + Cri) + (1//4)*(Cir + Crr)))*(-(1//4)*(Cii + Cri) + (1//4)*(Cir + Crr)) + (1//4)*(-(1//4)*((1//4)*(CiB + Cii + Cir + CrB + Cri + Crr) + CriB)*(Cii + Cir + Cri + Crr) - ((1//4)*(CiB + Cii + Cri) - (1//4)*(Cir + CrB + Crr))*(-(1//4)*(Cii + Cri) + (1//4)*(Cir + Crr)))*(Cii + Cir + Cri + Crr))) / (6.62607015e-34((-(1//4)*(CiA + Cii + Cir) + (1//4)*(CrA + Cri + Crr))*(((1//4)*(CiA + Cii + Cir) - (1//4)*(CrA + Cri + Crr))*(-(((1//4)*(CiB + Cii + Cri) - (1//4)*(Cir + CrB + Crr))^2) + (1//4)*((1//4)*(CiB + Cii + Cir + CrB + Cri + Crr) + CriB)*(CiB + Cii + Cir + CrB + Cri + Crr)) + (-(1//4)*((1//4)*(CiB + Cii + Cri) - (1

The above expressions are symbolic. To Julia, they represent functions for how to compute EC, EL. Once actual values are provided for the symbols, the abstract formulas are compiled down to machine-code as functions of the abstract symbols, with concrete types provided by the given values. Since the symbolic formulas are compiled to machine code, this allows for calculations of derivatives for anything down the line that utilizes these functions. This also allows for highly optimized sweeps over capacitiance and inductance values. Below we subsitute an example of capacitance and inductance values, and compile the abstract formulas to yield a numerical result.

In [15]:
# Capacitances (fF) of the circuit in the marimo notebook `xonium-taichi-couplers`: ground capacitances
# of 20/80/20/80 fF, 10 fF inside each fluxonium and a single 5 fF coupler between the two ring nodes.
# The cross-capacitances (and the island-island one) are zero. Inductances (nH) are example values:
# 100 nH for each fluxonium inductor and 1e9 nH for the ring-ring inductor, i.e. effectively none,
# since the marimo circuit has no inductor between the two fluxoniums.
params = [CrA, CiA, CriA, CrB, CiB, CriB, Crr, Cii, Cir, Cri, LriA, LriB, Lrr]
values = [80.0, 20.0, 10.0, 80.0, 20.0, 10.0, 5.0, 0.0, 0.0, 0.0, 100.0, 100.0, 1e9]

# compile the symbolic formulas above into one function of the parameters (no derivatives are taken)
names = (:ECA, :ECB, :ELA, :ELB, :JAB)
formulas_fn, _ = build_function([ECA, ECB, ELA, ELB, JAB], params; expression=Val(false))

# E_C and E_L in GHz, J in MHz. The marimo notebook gives E_C = 0.7399 GHz and J = 41.10 MHz
NamedTuple{names}(Tuple(formulas_fn(values)))

(ECA = 0.7398708283158923, ECB = 0.7398708283158923, ELA = 1.6346151689331905, ELB = 1.6346151689331905, JAB = 41.103934906438454)